In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [2]:
# Load raw data (we will preprocess quickly here)
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
credit_df = pd.read_csv('../data/raw/creditcard.csv')

print("Loaded datasets successfully.")

Loaded datasets successfully.


In [5]:
# === Fraud Dataset Modeling ===
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')

fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['time_since_signup'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds() / 3600
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour

# Use more features
X_f = fraud_df[['purchase_value', 'age', 'time_since_signup', 'hour_of_day']]
y_f = fraud_df['class']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_f, y_f, test_size=0.2, stratify=y_f, random_state=42
)

# Scale + SMOTE
scaler = StandardScaler()
X_train_f_scaled = scaler.fit_transform(X_train_f)
X_test_f_scaled = scaler.transform(X_test_f)

smote = SMOTE(random_state=42)
X_train_f_res, y_train_f_res = smote.fit_resample(X_train_f_scaled, y_train_f)

# Logistic Regression Baseline
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_f_res, y_train_f_res)
pred_lr = lr.predict(X_test_f_scaled)
prob_lr = lr.predict_proba(X_test_f_scaled)[:, 1]

print("=== Logistic Regression (Fraud Data) ===")
print(classification_report(y_test_f, pred_lr))
print("AUC-PR Score:", average_precision_score(y_test_f, prob_lr))

# XGBoost Model
xgb = XGBClassifier(random_state=42, eval_metric='aucpr', n_estimators=200, learning_rate=0.1)
xgb.fit(X_train_f_res, y_train_f_res)
pred_xgb = xgb.predict(X_test_f_scaled)
prob_xgb = xgb.predict_proba(X_test_f_scaled)[:, 1]

print("\n=== XGBoost (Fraud Data) ===")
print(classification_report(y_test_f, pred_xgb))
print("AUC-PR Score:", average_precision_score(y_test_f, prob_xgb))

=== Logistic Regression (Fraud Data) ===
              precision    recall  f1-score   support

           0       0.95      0.64      0.76     27393
           1       0.17      0.70      0.27      2830

    accuracy                           0.64     30223
   macro avg       0.56      0.67      0.52     30223
weighted avg       0.88      0.64      0.72     30223

AUC-PR Score: 0.5906878004916584

=== XGBoost (Fraud Data) ===
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     27393
           1       0.95      0.53      0.68      2830

    accuracy                           0.95     30223
   macro avg       0.95      0.76      0.83     30223
weighted avg       0.95      0.95      0.95     30223

AUC-PR Score: 0.605988408169901


In [6]:
# Credit Card Dataset
X_c = credit_df.drop('Class', axis=1)
y_c = credit_df['Class']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, stratify=y_c, random_state=42)

X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)

X_train_c_res, y_train_c_res = smote.fit_resample(X_train_c_scaled, y_train_c)

# Logistic Regression
lr_c = LogisticRegression(max_iter=1000, random_state=42)
lr_c.fit(X_train_c_res, y_train_c_res)
pred_lr_c = lr_c.predict(X_test_c_scaled)
prob_lr_c = lr_c.predict_proba(X_test_c_scaled)[:, 1]

print("=== Logistic Regression (Credit) ===")
print(classification_report(y_test_c, pred_lr_c))
print("AUC-PR:", average_precision_score(y_test_c, prob_lr_c))

# XGBoost
xgb_c = XGBClassifier(random_state=42, eval_metric='aucpr')
xgb_c.fit(X_train_c_res, y_train_c_res)
pred_xgb_c = xgb_c.predict(X_test_c_scaled)
prob_xgb_c = xgb_c.predict_proba(X_test_c_scaled)[:, 1]

print("\n=== XGBoost (Credit) ===")
print(classification_report(y_test_c, pred_xgb_c))
print("AUC-PR:", average_precision_score(y_test_c, prob_xgb_c))

=== Logistic Regression (Credit) ===
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962

AUC-PR: 0.724469435669471

=== XGBoost (Credit) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.73      0.85      0.78        98

    accuracy                           1.00     56962
   macro avg       0.86      0.92      0.89     56962
weighted avg       1.00      1.00      1.00     56962

AUC-PR: 0.8682821332846721
